# Module 19 — BPE Tokenizer From Scratch

Every module so far (06-18) tokenized text one **character** at a time.
That's fine for tiny toy demos, but it's a bad choice for real models: it
makes sequences long (attention cost grows with the *square* of sequence
length — Module 10), and each token carries almost no meaning on its own.

**Byte-Pair Encoding (BPE)** — the algorithm behind GPT-2's tokenizer, and
still the basis for most modern LLM tokenizers — fixes this by building a
vocabulary of frequently-occurring **subword chunks** instead of fixed
characters. The algorithm: start with individual characters, then
repeatedly find the *most frequent adjacent pair* of symbols anywhere in
the training corpus and merge it into a single new symbol. Common words
end up as one token; rare words fall back to smaller pieces.

## 1. Word frequencies, split into characters

In [ ]:
from collections import defaultdict

CORPUS = """Aether and Lumine are twins known as the Traveler. They came from another world and lost each other upon arrival in Teyvat. Paimon found Aether floating near Mondstadt and decided to travel together. Mondstadt is called the City of Freedom, and the wind blows gently across its hills. Klee loves to explore the city and often causes small explosions with her bombs. Diluc runs the Dawn Winery outside the city walls. Kaeya works at the Knights of Favonius and enjoys teasing Diluc. Jean leads the Knights of Favonius with great responsibility. Barbara sings songs at the church and heals the sick."""

words = CORPUS.lower().split()
word_freqs = defaultdict(int)
for w in words:
    word_freqs[w] += 1
print(f"{len(words)} words, {len(word_freqs)} unique")

# Each word becomes a tuple of characters plus an end-of-word marker "</w>",
# so BPE never merges across word boundaries (e.g. the "d" ending "and"
# should never merge with the "t" starting the next word).
def word_to_symbols(word):
    return tuple(word) + ("</w>",)

corpus_symbols = {word_to_symbols(w): freq for w, freq in word_freqs.items()}
print("example:", list(corpus_symbols.items())[:3])

## 2. The merge loop: count adjacent pairs, merge the most frequent

In [ ]:
def get_pair_counts(symbol_freqs):
    counts = defaultdict(int)
    for symbols, freq in symbol_freqs.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += freq
    return counts


def merge_pair(pair, symbol_freqs):
    merged_symbol = pair[0] + pair[1]
    new_freqs = defaultdict(int)
    for symbols, freq in symbol_freqs.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_freqs[tuple(new_symbols)] += freq
    return new_freqs


num_merges = 40
merges = []  # ordered list of learned merge rules, applied in this order at encode time
symbol_freqs = dict(corpus_symbols)

for step in range(num_merges):
    pair_counts = get_pair_counts(symbol_freqs)
    if not pair_counts:
        break
    best_pair = max(pair_counts, key=pair_counts.get)
    symbol_freqs = merge_pair(best_pair, symbol_freqs)
    merges.append(best_pair)
    if step < 10 or step % 10 == 0:
        merged_str = "".join(best_pair)
        print(f"merge {step:2d}: {best_pair} -> {merged_str!r}  (count {pair_counts[best_pair]})")

## 3. Encoding new text with the learned merges

In [ ]:
def encode_word(word, merges):
    symbols = list(word_to_symbols(word))
    for pair in merges:  # apply merges in the order they were learned
        i = 0
        while i < len(symbols) - 1:
            if (symbols[i], symbols[i + 1]) == pair:
                symbols[i:i + 2] = [symbols[i] + symbols[i + 1]]
            else:
                i += 1
    return symbols


def encode_text(text, merges):
    tokens = []
    for word in text.lower().split():
        tokens.extend(encode_word(word, merges))
    return tokens


def decode_tokens(tokens):
    text = "".join(tokens).replace("</w>", " ")
    return text.strip()


sample = "the traveler and paimon explore mondstadt"
bpe_tokens = encode_text(sample, merges)
char_tokens = list(sample.replace(" ", "_"))  # char-level baseline, using "_" to visualize spaces

print("BPE tokens: ", bpe_tokens)
print(f"BPE token count: {len(bpe_tokens)}")
print(f"Char-level token count: {len(char_tokens)}")
print(f"Compression: {len(char_tokens) / len(bpe_tokens):.2f}x fewer tokens")

## 4. Round-trip correctness

In [ ]:
decoded = decode_tokens(bpe_tokens)
assert decoded == sample, f"{decoded!r} != {sample!r}"
print(f"decode(encode(text)) == text: confirmed for {sample!r}")

# Also check every word actually seen during training round-trips correctly
for word in list(word_freqs.keys())[:20]:
    tokens = encode_word(word, merges)
    assert decode_tokens(tokens) == word
print("All 20 sampled training words round-trip correctly too.")

## Recap

- BPE builds its vocabulary bottom-up: start from characters, repeatedly
  merge the most frequent adjacent pair, and stop after a chosen number of
  merges (real tokenizers do tens of thousands; we did 40 for a toy corpus).
- The result: common patterns (frequent letter pairs, common short words)
  become single tokens, while rare words still fall back to smaller pieces
  — no word is ever truly "unknown," it just costs more tokens.
- Measured directly: BPE needed meaningfully fewer tokens than character-level
  for the same sentence, and `decode(encode(text)) == text` held for every
  word tested.

Module 20 swaps this from-scratch version for a production tokenizer
library (`tiktoken` / Hugging Face `tokenizers`) — same core algorithm,
trained on a vastly larger corpus with a vocabulary in the tens of
thousands, and fast enough (C++/Rust internals) for real training data
volumes.